# OOP — Senior Python Interview Practice\n\nFour interview-style exercises spanning implementation, trade-offs, and production concerns.\n\n**How to use:** attempt each prompt first, then run and critique the reference solution. Discuss trade-offs aloud as you would in a senior-level interview.


## 1. Immutable chat message\n\n### Problem statement\nCreate a validated immutable value object for a chat message.


In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True, slots=True)
class Message:
    role: str
    content: str
    def __post_init__(self):
        if self.role not in {'system', 'user', 'assistant'}:
            raise ValueError('invalid role')
        if not self.content.strip(): raise ValueError('content is required')

Message('user', 'Explain RAG')


### Complexity\n- **Time:** O(length of content)\n- **Space:** O(1) auxiliary\n\n### Interview tip\nValue objects benefit from immutability: safer caching, hashing, and concurrency.\n\n### Follow-up questions\n- When would you avoid frozen dataclasses? How would you add metadata?


## 2. Pluggable retriever\n\n### Problem statement\nDefine an abstract retriever contract and two implementations.


In [ ]:
from abc import ABC, abstractmethod

class Retriever(ABC):
    @abstractmethod
    def search(self, query, k=3): ...

class InMemoryRetriever(Retriever):
    def __init__(self, docs): self.docs = docs
    def search(self, query, k=3):
        terms = set(query.lower().split())
        return sorted(self.docs, key=lambda d: len(terms & set(d.lower().split())), reverse=True)[:k]


### Complexity\n- **Time:** Depends on implementation\n- **Space:** Depends on index\n\n### Interview tip\nCenter the interface on caller needs, not storage details.\n\n### Follow-up questions\n- Where do filters, scores, and async support belong in the contract?


## 3. Resource context manager\n\n### Problem statement\nImplement a context manager that opens a file-like resource and guarantees cleanup.


In [ ]:
class ManagedResource:
    def __init__(self, opener, closer): self.opener, self.closer = opener, closer
    def __enter__(self): self.resource = self.opener(); return self.resource
    def __exit__(self, exc_type, exc, tb):
        self.closer(self.resource)
        return False  # propagate errors


### Complexity\n- **Time:** O(1) management overhead\n- **Space:** O(1)\n\n### Interview tip\nSay whether exceptions should be suppressed; defaulting to propagation is safest.\n\n### Follow-up questions\n- How would this differ for async resources?


## 4. Dependency injection\n\n### Problem statement\nModel an agent service that receives its model client rather than creating one internally.


In [ ]:
class AgentService:
    def __init__(self, model_client): self.model_client = model_client
    def answer(self, prompt): return self.model_client.generate(prompt)

class FakeModel:
    def generate(self, prompt): return f'fake: {prompt}'

AgentService(FakeModel()).answer('hello')


### Complexity\n- **Time:** Model-call dependent\n- **Space:** O(1) auxiliary\n\n### Interview tip\nInjected dependencies make testing deterministic and keep infrastructure out of domain logic.\n\n### Follow-up questions\n- How do you wire real dependencies at application startup?
